Этап 2. Загрузка и первичный осмотр данных

**Источник:** Natural Resources Canada, Fuel Consumption Ratings (open.canada.ca), лицензия Open Government Licence Canada.
**Файл:** `my2015-2024-fuel-consumption-ratings.csv`: основной файл для обучения и тестирования. Файлы 1995-2014, 2025 и 2026 не смешиваем с основным.

**Что делаем:** читаем CSV в таблицу `df` через `pd.read_csv`.
- Путь относительный (`../data/raw/...`), чтобы ноутбук запускался у всех участников команды.
- `na_values=["n/a"]`: текст `n/a` читается как настоящий пропуск (NaN), а не как строка.

In [7]:
import pandas as pd

# Путь относительно папки notebooks/
path = "../data/raw/my2015-2024-fuel-consumption-ratings.csv"

# na_values: чтобы текст "n/a" читался как настоящий пропуск (NaN), а не как строка
df = pd.read_csv(path, na_values=["n/a"])

### Размер данных

`df.shape` показывает число строк и столбцов.

**Результат:** (10060, 15), то есть 10060 автомобилей и 15 столбцов.
**Вывод:** для Linear Regression, Decision Tree и KNN этого достаточно, модели обучаются быстро.

In [8]:
print(df.shape)

(10060, 15)


### Типы данных и пропуски

`df.info()` показывает тип каждого столбца и сколько в нём непустых значений.

**Результат:**
- Типы: 6 столбцов float64, 4 int64, 5 текстовых.
- Пропусков нет везде, кроме двух столбцов: `CO2 rating` (8932 из 10060 заполнены) и `Smog rating` (7826 из 10060).

**Вывод:** оба столбца с пропусками мы всё равно не берём в признаки из-за утечки данных (см. ниже), поэтому заполнять пропуски не нужно.

In [9]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10060 entries, 0 to 10059
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model year            10060 non-null  int64  
 1   Make                  10060 non-null  str    
 2   Model                 10060 non-null  str    
 3   Vehicle class         10060 non-null  str    
 4   Engine size (L)       10060 non-null  float64
 5   Cylinders             10060 non-null  int64  
 6   Transmission          10060 non-null  str    
 7   Fuel type             10060 non-null  str    
 8   City (L/100 km)       10060 non-null  float64
 9   Highway (L/100 km)    10060 non-null  float64
 10  Combined (L/100 km)   10060 non-null  float64
 11  Combined (mpg)        10060 non-null  int64  
 12  CO2 emissions (g/km)  10060 non-null  int64  
 13  CO2 rating            8932 non-null   float64
 14  Smog rating           7826 non-null   float64
dtypes: float64(6), int64(4), str(5

### Первые строки

`df.head()` показывает первые 5 строк, чтобы увидеть, как выглядят данные.

**Что видно:**
- Данные начинаются с 2015 года, значит, загружен правильный файл.
- `Vehicle class` записан в разных форматах (например, `Compact` и `Sport utility vehicle: Small`): категории нужно привести к единому виду.
- `Transmission` состоит из типа и числа передач (`AS5`, `M6`, `AV7`): из неё можно сделать два признака.
- `Fuel type` закодирован буквами.
- В названии модели встречаются гибриды (например, `ILX Hybrid`).

In [10]:
print(df.head())

   Model year   Make       Model                 Vehicle class  \
0        2015  Acura         ILX                       Compact   
1        2015  Acura         ILX                       Compact   
2        2015  Acura  ILX Hybrid                       Compact   
3        2015  Acura  MDX SH-AWD  Sport utility vehicle: Small   
4        2015  Acura     RDX AWD  Sport utility vehicle: Small   

   Engine size (L)  Cylinders Transmission Fuel type  City (L/100 km)  \
0              2.0          4          AS5         Z              9.7   
1              2.4          4           M6         Z             10.8   
2              1.5          4          AV7         Z              6.0   
3              3.5          6          AS6         Z             12.7   
4              3.5          6          AS6         Z             12.1   

   Highway (L/100 km)  Combined (L/100 km)  Combined (mpg)  \
0                 6.7                  8.3              34   
1                 7.4                  9

### Список столбцов

`df.columns.tolist()` выводит точные названия столбцов, чтобы обращаться к ним в коде без опечаток.

**Кандидаты в признаки:** `Model year`, `Make`, `Model`, `Vehicle class`, `Engine size (L)`, `Cylinders`, `Transmission`, `Fuel type`.

**Target:** `Combined (L/100 km)`.

**Исключаем из признаков (data leakage):** `City (L/100 km)`, `Highway (L/100 km)`, `Combined (mpg)`, `CO2 emissions (g/km)`, `CO2 rating`, `Smog rating`. Они напрямую выражают target или считаются из него, и модель получила бы ответ вместо того, чтобы его предсказывать.

In [11]:
print(df.columns.tolist())

['Model year', 'Make', 'Model', 'Vehicle class', 'Engine size (L)', 'Cylinders', 'Transmission', 'Fuel type', 'City (L/100 km)', 'Highway (L/100 km)', 'Combined (L/100 km)', 'Combined (mpg)', 'CO2 emissions (g/km)', 'CO2 rating', 'Smog rating']


### Диапазон лет, типы топлива и статистика

В ячейке три проверки:
1. Минимум и максимум года. **Результат:** 2015 и 2024, файл верный.
2. `value_counts()` по `Fuel type`. **Результат:** X: 4756, Z: 4739, E: 328, D: 236, N: 1. Категории очень разной частоты, а `N` встречается всего один раз (решение по нему примем при очистке). Расшифровку букв сверим с документацией источника.
3. `describe()`: минимум, максимум, среднее по числовым столбцам.

**Что видно в `describe()`:**
- Target `Combined (L/100 km)`: от 4.0 до 26.1, среднее около 11.0, std около 2.8. Разброс достаточный, чтобы было что предсказывать.
- Объём двигателя 0.9-8.4 л, цилиндры 3-16: значения правдоподобны, явных ошибок не видно.

**Ограничение датасета (limitation):** в данных нет веса и мощности автомобиля.

In [12]:
print(df["Model year"].min(), df["Model year"].max())
print(df["Fuel type"].value_counts())
df.describe()

2015 2024
Fuel type
X    4756
Z    4739
E     328
D     236
N       1
Name: count, dtype: int64


,Model year,Engine size (L),Cylinders,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
count,10060.000000,10060.000000,10060.00000,10060.000000,10060.000000,10060.000000,10060.000000,10060.000000,8932.000000,7826.000000
mean,2019.242048,3.144901,5.61839,12.483052,9.142803,10.979712,27.436282,253.711332,4.617443,4.816509
std,2.834109,1.340010,1.87591,3.404483,2.178649,2.808468,7.316270,60.474917,1.574763,1.783793
min,2015.000000,0.900000,3.00000,4.000000,3.900000,4.000000,11.000000,94.000000,1.000000,1.000000
25%,2017.000000,2.000000,4.00000,10.100000,7.600000,9.000000,22.000000,210.000000,4.000000,3.000000
50%,2019.000000,3.000000,6.00000,12.100000,8.900000,10.700000,26.000000,250.000000,5.000000,5.000000
75%,2022.000000,3.700000,6.00000,14.500000,10.400000,12.700000,31.000000,292.000000,5.000000,6.000000
max,2024.000000,8.400000,16.00000,30.700000,20.900000,26.100000,71.000000,608.000000,10.000000,8.000000


# Этап 3. Очистка данных
Проверяем: дубликаты, пропуски, диапазоны, категории, подозрительные признаки (leakage).
Каждое решение записываем с причиной.

## 3.1 Дубликаты

**Что проверяем:** есть ли в таблице полностью одинаковые строки (совпадают все 15 столбцов).
**Зачем:** повторяющиеся строки учили бы модель на одних и тех же данных несколько раз и делали бы оценку качества менее честной.
**Как:** `df.duplicated().sum()` считает повторные копии строк.

In [6]:
# Сколько полностью одинаковых строк (вторая и последующие копии)
print("Дубликатов:", df.duplicated().sum())
# Показываем все копии, чтобы понять, ошибка это или реальные машины
df[df.duplicated(keep=False)].sort_values(["Make", "Model", "Model year"]).head(10)

NameError: name 'df' is not defined